In [ ]:
# --- MASTER INTERACTIVE GUI (S-CORP SHAREHOLDER TAXATION MATRIX) ---
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

# Global Database Configurations
DB_NAME = "payroll_ledger.db"
CSV_BACKUP_NAME = "payroll_history_backup.csv"
EMPLOYEE_ID = "EE-SCORP-01"

# 2026 Statutory Caps & Standard Match Targets
SS_WAGE_BASE = 184500.00      
FUTA_WAGE_BASE = 7000.00      
NM_SUI_WAGE_BASE = 34800.00   
SS_RATE, MEDICARE_RATE, FUTA_NET_RATE, NM_SUI_RATE = 0.062, 0.0145, 0.006, 0.034

# 2026 Progressive Income Tax Tables 
TAX_TABLES_2026 = {
    'Single': {
        'FED': [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)],
        'NM':  [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]
    },
    'Married Filing Jointly': {
        'FED': [(24800, 0.10), (100800, 0.12), (211400, 0.22), (403550, 0.24), (512450, 0.32), (768700, 0.35), (float('inf'), 0.37)],
        'NM':  [(16100, 0.00), (27100, 0.015), (41100, 0.032), (49100, 0.032), (66000, 0.043), (82000, 0.043), (116000, 0.047), (148000, 0.047), (204200, 0.047), (232200, 0.049), (435000, 0.049), (662200, 0.049), (float('inf'), 0.059)]
    }
}

# 1. Setup Form Layout Controls (Including added pay period slider)
status_dropdown = widgets.Dropdown(options=['Single', 'Married Filing Jointly'], value='Single', description='Filing Status:')
salary_slider = widgets.IntSlider(value=60000, min=30000, max=250000, step=1000, description='Annual Base:')
periods_slider = widgets.IntSlider(value=12, min=4, max=52, step=1, description='Pay Periods:')
hip_slider = widgets.FloatSlider(value=1310.0, min=0.0, max=2500.0, step=10.0, description='S-Corp HIP:')
retire_slider = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5, description='401k Def %:')
date_picker = widgets.DatePicker(value=pd.to_datetime('2026-06-15').date(), description='Pay Date:')
save_button = widgets.Button(description="Commit Pay Run", button_style='success', icon='check')
output_panel = widgets.Output()

# 2. Computational Math Functions
def calc_capped(gross, ytd, cap, rate):
    if ytd >= cap: return 0.0
    return min(gross, cap - ytd) * rate

def calc_progressive(gross, periods, brackets):
    annual = gross * periods
    tax = 0.0
    prev = 0.0
    for limit, rate in brackets:
        if annual > limit:
            tax += (limit - prev) * rate
            prev = limit
        else:
            tax += (annual - prev) * rate
            break
    return tax / periods

# 3. GUI Visualization Refresh Loop
def update_dashboard(*args):
    with output_panel:
        clear_output(wait=True)
        
        # Calculate YTD from historical ledger tables cleanly
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
        res = cursor.fetchone()
        ytd_prior = float(res[0]) if res and res[0] is not None else 0.00
        conn.close()

        
        # Extract slider control selections
        status = status_dropdown.value
        periods = periods_slider.value
        base_gross = salary_slider.value / periods
        hip_reimbursement = hip_slider.value
        
        # Total payout cash received by owner
        total_period_payout = base_gross + hip_reimbursement
        deduction_401k = base_gross * (retire_slider.value / 100.0)
        
        # S-CORP STRUCTURAL SEPARATION: 
        # FICA applies to base gross pay only. HIP is exempt.
        fica_subject_gross = base_gross 
        # Income Tax applies to base gross AND health premiums minus pre-tax deductions
        income_tax_subject_gross = base_gross + hip_reimbursement - deduction_401k
        
        # Pull targeted filing structures
        selected_fed_brackets = TAX_TABLES_2026[status]['FED']
        selected_nm_brackets = TAX_TABLES_2026[status]['NM']
        
        # Execute tax calculations
        ee_ss = calc_capped(fica_subject_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
        ee_med = fica_subject_gross * MEDICARE_RATE
        ee_fed = calc_progressive(income_tax_subject_gross, periods, selected_fed_brackets)
        ee_nm = calc_progressive(income_tax_subject_gross, periods, selected_nm_brackets)
        
        nm_wc_fee_per_pay = (2.55 * 4) / periods
        
        # Calculate Owner Take-Home Net Cash
        total_ee_taxes = ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee_per_pay
        net_pay = total_period_payout - deduction_401k - total_ee_taxes
        
        # Chart Generation
        categories = ['Net Take-Home', 'Pre-Tax 401k', 'Fed Income Tax', 'NM State Tax', 'FICA taxes']
        sizes = [net_pay, deduction_401k, ee_fed, ee_nm, (ee_ss + ee_med)]
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#95a5a6']
        
        combined_labels = [f"{cat}\n(${val:,.2f})" for cat, val in zip(categories, sizes)]
        labels = [l for i, l in enumerate(combined_labels) if sizes[i] > 0]
        colors = [c for i, c in enumerate(colors) if sizes[i] > 0]
        sizes = [s for s in sizes if s > 0]
        
        fig, ax = plt.subplots(figsize=(8.5, 4.5))
        ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, pctdistance=0.65, labeldistance=1.15, textprops={'fontsize': 9, 'weight': 'bold'})
        ax.set_title(f"S-Corp Shareholder Allocation [{status.upper()}] (Total Payout: ${total_period_payout:,.2f})", fontsize=11, weight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        
        print(f"💰 Owner Net Cash: ${net_pay:,.2f} | 🩺 Taxable S-Corp HIP Amount: ${hip_reimbursement:,.2f}")
        print(f"ℹ️ Base Wages (FICA Subject): ${base_gross:,.2f} | Income Tax Subject Base: ${income_tax_subject_gross:,.2f}")

# 4. Action Handle: Append Ledger to Local Database File
def on_save_clicked(b):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
    res = cursor.fetchone()
    ytd_prior = float(res[0]) if res and res[0] is not None else 0.00
    
    status = status_dropdown.value
    periods = periods_slider.value
    base_gross = salary_slider.value / periods
    hip_reimbursement = hip_slider.value
    current_gross = base_gross + hip_reimbursement
    deduction_401k = base_gross * (retire_slider.value / 100.0)
    
    fica_subject_gross = base_gross
    income_tax_subject_gross = base_gross + hip_reimbursement - deduction_401k
    
    ee_ss = calc_capped(fica_subject_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
    ee_med = fica_subject_gross * MEDICARE_RATE
    ee_fed = calc_progressive(income_tax_subject_gross, periods, TAX_TABLES_2026[status]['FED'])
    ee_nm = calc_progressive(income_tax_subject_gross, periods, TAX_TABLES_2026[status]['NM'])
    nm_wc_fee = (2.55 * 4) / periods
    net_pay = current_gross - deduction_401k - (ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee)
    
    # Employer tax savings match employee tax savings on HIP exemption
    er_ss = ee_ss
    er_med = ee_med
    er_futa = calc_capped(fica_subject_gross, ytd_prior, FUTA_WAGE_BASE, FUTA_NET_RATE)
    er_nm_sui = calc_capped(fica_subject_gross, ytd_prior, NM_SUI_WAGE_BASE, NM_SUI_RATE)
    total_company_cost = current_gross + er_ss + er_med + er_futa + er_nm_sui + nm_wc
    
    cursor.execute("""
    INSERT INTO payroll_ledger (
        employee_id, pay_period_ending, gross_wages, pre_tax_401k, ee_ss, ee_medicare, ee_fed_tax, ee_nm_tax, ee_wc_fee, net_pay,
        er_ss, er_medicare, er_futa, er_nm_sui, er_wc_fee, total_company_cost
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        EMPLOYEE_ID, str(date_picker.value), current_gross, deduction_401k, ee_ss, ee_med, ee_fed, ee_nm, nm_wc, net_pay,
        er_ss, er_med, er_futa, er_nm_sui, nm_wc, total_company_cost
    ))
    conn.commit()
    df = pd.read_sql_query("SELECT * FROM payroll_ledger", conn)
    df.to_csv(CSV_BACKUP_NAME, index=False)
    conn.close()
    
    update_dashboard()
    with output_panel:
        print(f"\n✅ SUCCESS: S-Corp Shareholder payroll run committed safely.")

# Attach unified runtime change observers
status_dropdown.observe(update_dashboard, 'value')
salary_slider.observe(update_dashboard, 'value')
periods_slider.observe(update_dashboard, 'value')
hip_slider.observe(update_dashboard, 'value')
retire_slider.observe(update_dashboard, 'value')
date_picker.observe(update_dashboard, 'value')
save_button.on_click(on_save_clicked)

# Render Controls Group
input_form = widgets.VBox([status_dropdown, salary_slider, periods_slider, hip_slider, retire_slider, date_picker, save_button])
dashboard_layout = widgets.HBox([input_form, output_panel])
display(dashboard_layout)

update_dashboard()
